# PCA-denoised STEM-EDX composition mapping

**Purpose.** Reconstruct a STEM-EDX spectrum image from selected
principal components, extract background-corrected X-ray intensities,
quantify atomic composition using the Cliff–Lorimer method, and construct
guarded elemental-ratio maps.

**Input.** A HyperSpy-readable EDS-TEM spectrum image (`.hspy`, `.emd` or
another supported format).

**Output.** PCA diagnostics, elemental composition maps, optional ratio
maps and optional line profiles.

This notebook is a compact method record for experienced STEM-EDX users.
Raw datasets are not distributed with the repository.

## 1. Setup and parameters

All paths and experiment-specific choices are defined here. Verify the
energy calibration, X-ray lines, background windows and Cliff–Lorimer
factors for the microscope and specimen before interpreting results.

In [ ]:
%matplotlib inline

from importlib.metadata import version
from pathlib import Path

import exspy  # Registers current EDS/EELS signal extensions with HyperSpy.
import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np

plt.set_cmap("magma")
print("HyperSpy:", hs.__version__)
print("eXSpy:", version("exspy"))

In [ ]:
# Repository-relative paths. This works when Jupyter starts from either
# the repository root or its notebooks/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data/perovskite_edx_spectrum_image.hspy"
OUTPUT_DIR = REPO_ROOT / "outputs/edx_composition"

# Loading
# If hs.load() returns several signals from an EMD file, select one signal
# by index or sum the last N EDS detector signals.
INPUT_SIGNAL_INDEX = None
SUM_LAST_N_EDS_SIGNALS = None
LOAD_LAZY = True

# Preprocessing
ENERGY_CROP_CHANNELS = (70, 3000)  # Integer channel indices; set None to disable.
REBIN_SCALE = (4, 4, 4)            # navigation x, navigation y, energy
SIGNAL_TITLE = "PCA-denoised perovskite STEM-EDX"

# PCA/SVD reconstruction
PCA_COMPONENTS = (0, 1, 2, 3, 4, 5)
EXPLAINED_VARIANCE_COMPONENTS = 20

# Quantified X-ray lines. HyperSpy/eXSpy returns line intensities in
# alphabetical line order; factors are matched by line name below.
ELEMENTS = ("Ag", "C", "Cl", "I", "In", "N", "O", "Pb", "Sn")
XRAY_LINES = (
    "Ag_La",
    "C_Ka",
    "Cl_Ka",
    "I_La",
    "In_La",
    "N_Ka",
    "O_Ka",
    "Pb_La",
    "Sn_La",
)

# Manual background windows in keV: pre-start, pre-end, post-start, post-end.
BACKGROUND_WINDOWS_KEV = {
    "Ag_La": (2.00, 2.10, 5.30, 5.50),
    "C_Ka":  (0.12, 0.16, 0.66, 0.70),
    "Cl_Ka": (2.00, 2.10, 2.75, 2.85),
    "I_La":  (2.00, 2.10, 5.30, 5.50),
    "In_La": (2.00, 2.10, 5.30, 5.50),
    "N_Ka":  (0.12, 0.16, 0.66, 0.70),
    "O_Ka":  (0.12, 0.16, 0.66, 0.70),
    "Pb_La": (9.70, 10.20, 11.00, 11.50),
    "Sn_La": (2.00, 2.10, 5.30, 5.50),
}
INTEGRATION_WINDOW_FACTOR = 2.2

# Original Spectra Cliff–Lorimer factors used in the source workflow.
# Replace these with the calibrated values appropriate to the experiment.
K_FACTORS = {
    "Ag_La": 0.480,
    "C_Ka":  0.533,
    "Cl_Ka": 0.323,
    "I_La":  0.492,
    "In_La": 8.400,
    "N_Ka":  0.421,
    "O_Ka":  0.368,
    "Pb_La": 0.824,
    "Sn_La": 0.489,
}
COMPOSITION_UNITS = "atomic"

# Minimum atomic-percent values used before ratio calculation.
MAP_THRESHOLDS = {
    "C_Ka": 0.5,
    "Cl_Ka": 0.1,
    "I_La": 0.5,
    "N_Ka": 0.5,
    "O_Ka": 0.5,
    "Pb_La": 0.5,
    "Sn_La": 0.1,
}

# name: numerator line, denominator line, display minimum, display maximum
RATIO_DEFINITIONS = {
    "I_over_Pb": ("I_La", "Pb_La", 1.0, 5.0),
    "Pb_over_I": ("Pb_La", "I_La", 0.1, 0.7),
    "Pb_over_Sn": ("Pb_La", "Sn_La", 0.1, 3.0),
    "C_over_N": ("C_Ka", "N_Ka", 1.0, 8.0),
    "C_over_O": ("C_Ka", "O_Ka", 0.5, 4.0),
    "N_over_O": ("N_Ka", "O_Ka", 0.5, 4.0),
    "I_over_Cl": ("I_La", "Cl_Ka", 1.0, 15.0),
}

# Optional profile through one ratio map.
LINE_PROFILE_RATIO = "I_over_Pb"
LINE_PROFILE_AXIS = 0
LINE_PROFILE_INDEX = 22
LINE_PROFILE_START = 0
LINE_PROFILE_STOP = 73
LINE_PROFILE_STEP_NM = None

SAVE_RESULTS = False

## 2. Load and identify the EDS spectrum image

HyperSpy can return a single signal or a list of signals from container
formats such as Velox EMD. The selection options above make detector
combination explicit instead of relying on hard-coded list positions.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Update DATA_PATH in the parameter cell."
    )

loaded = hs.load(DATA_PATH, lazy=LOAD_LAZY)

if isinstance(loaded, (list, tuple)):
    if SUM_LAST_N_EDS_SIGNALS is not None:
        if SUM_LAST_N_EDS_SIGNALS < 1 or SUM_LAST_N_EDS_SIGNALS > len(loaded):
            raise ValueError("SUM_LAST_N_EDS_SIGNALS is outside the loaded list.")
        detector_signals = list(loaded[-SUM_LAST_N_EDS_SIGNALS:])
        spectrum_image = detector_signals[0]
        for detector_signal in detector_signals[1:]:
            spectrum_image = spectrum_image + detector_signal
    elif INPUT_SIGNAL_INDEX is not None:
        spectrum_image = loaded[INPUT_SIGNAL_INDEX]
    else:
        raise ValueError(
            "The input contains multiple signals. Set INPUT_SIGNAL_INDEX "
            "or SUM_LAST_N_EDS_SIGNALS in the parameter cell."
        )
else:
    spectrum_image = loaded

spectrum_image.set_signal_type("EDS_TEM")
print(spectrum_image)
print(spectrum_image.axes_manager)

## 3. Crop, rebin and materialize the signal

Cropping and rebinning are applied before loading lazy data into memory.
Confirm that the integer crop limits correspond to the intended energy
range for the imported calibration.

In [ ]:
if ENERGY_CROP_CHANNELS is not None:
    spectrum_image.crop(
        axis=-1,
        start=ENERGY_CROP_CHANNELS[0],
        end=ENERGY_CROP_CHANNELS[1],
    )

if REBIN_SCALE != (1, 1, 1):
    spectrum_image = spectrum_image.rebin(scale=REBIN_SCALE)

if getattr(spectrum_image, "_lazy", False):
    spectrum_image.compute()

spectrum_image.change_dtype("float32")
spectrum_image.metadata.set_item("General.title", SIGNAL_TITLE)
spectrum_image.set_signal_type("EDS_TEM")

spectrum_image.sum().plot()
print(spectrum_image.axes_manager)

## 4. PCA denoising

Principal components are calculated with Poisson-noise normalization.
The reconstruction uses the component indices declared above. Inspect the
scree plot and factors/loadings before changing this selection.

In [ ]:
spectrum_image.decomposition(normalize_poissonian_noise=True)
spectrum_image.plot_explained_variance_ratio(
    n=EXPLAINED_VARIANCE_COMPONENTS
)
spectrum_image.plot_decomposition_results()

In [ ]:
denoised = spectrum_image.get_decomposition_model(
    list(PCA_COMPONENTS)
)
denoised.set_signal_type("EDS_TEM")
denoised.metadata.set_item(
    "General.title",
    f"{SIGNAL_TITLE} | PCA components {PCA_COMPONENTS}",
)

denoised.sum().plot()

## 5. Define lines and background windows

eXSpy sorts extracted intensities alphabetically by X-ray line. Window
rows and Cliff–Lorimer factors are therefore matched by line name rather
than by fragile numeric positions.

In [ ]:
denoised.set_elements(list(ELEMENTS))
denoised.set_lines(list(XRAY_LINES))

background_windows = denoised.estimate_background_windows(
    line_width=[2.0, 2.0]
)
metadata_line_order = list(denoised.metadata.Sample.xray_lines)

missing_windows = set(metadata_line_order) - set(BACKGROUND_WINDOWS_KEV)
if missing_windows:
    raise KeyError(f"No background windows supplied for: {sorted(missing_windows)}")

for row_index, line in enumerate(metadata_line_order):
    background_windows[row_index, :] = BACKGROUND_WINDOWS_KEV[line]

denoised.sum().plot(
    background_windows=background_windows,
    integration_windows=INTEGRATION_WINDOW_FACTOR,
)
print("Line order:", metadata_line_order)

## 6. Extract line intensities and quantify composition

Negative fitted intensities are clipped to zero before Cliff–Lorimer
quantification. Factor ordering is derived from each intensity signal's
metadata.

In [ ]:
intensities = denoised.get_lines_intensity(
    background_windows=background_windows,
    integration_windows=INTEGRATION_WINDOW_FACTOR,
    plot_result=False,
)

intensity_lines = []
for intensity in intensities:
    line = list(intensity.metadata.Sample.xray_lines)[0]
    intensity_lines.append(line)
    intensity.data = np.clip(intensity.data, 0, None)

missing_factors = set(intensity_lines) - set(K_FACTORS)
if missing_factors:
    raise KeyError(f"No Cliff–Lorimer factors supplied for: {sorted(missing_factors)}")

ordered_factors = [K_FACTORS[line] for line in intensity_lines]
print("Quantification order:", list(zip(intensity_lines, ordered_factors)))

In [ ]:
composition_maps = denoised.quantification(
    intensities,
    method="CL",
    factors=ordered_factors,
    composition_units=COMPOSITION_UNITS,
    plot_result=False,
)
composition_by_line = dict(zip(intensity_lines, composition_maps))

hs.plot.plot_images(
    composition_maps,
    cmap="viridis",
    axes_decor="off",
    scalebar="all",
)

## 7. Construct guarded elemental-ratio maps

Ratios are calculated only where both composition maps exceed their
declared thresholds. Invalid or weak-signal pixels remain `NaN` instead
of being converted into apparently meaningful zeros.

In [ ]:
def guarded_ratio(
    numerator,
    denominator,
    numerator_threshold,
    denominator_threshold,
    title,
):
    numerator_data = np.asarray(numerator.data, dtype=float)
    denominator_data = np.asarray(denominator.data, dtype=float)
    valid = (
        (numerator_data >= numerator_threshold)
        & (denominator_data >= denominator_threshold)
    )

    ratio_data = np.full_like(numerator_data, np.nan, dtype=float)
    np.divide(
        numerator_data,
        denominator_data,
        out=ratio_data,
        where=valid,
    )

    ratio_signal = numerator.deepcopy()
    ratio_signal.data = ratio_data
    ratio_signal.metadata.set_item("General.title", title)
    if (
        ratio_signal.axes_manager.signal_dimension != 2
        and ratio_signal.data.ndim == 2
    ):
        ratio_signal = ratio_signal.as_signal2D((0, 1))
    return ratio_signal


ratio_maps = {}
for name, (numerator_line, denominator_line, vmin, vmax) in (
    RATIO_DEFINITIONS.items()
):
    if numerator_line not in composition_by_line:
        raise KeyError(f"{numerator_line} is not available for {name}.")
    if denominator_line not in composition_by_line:
        raise KeyError(f"{denominator_line} is not available for {name}.")

    ratio = guarded_ratio(
        composition_by_line[numerator_line],
        composition_by_line[denominator_line],
        MAP_THRESHOLDS.get(numerator_line, 0.0),
        MAP_THRESHOLDS.get(denominator_line, 0.0),
        name.replace("_", " "),
    )
    ratio_maps[name] = ratio
    ratio.plot(
        vmin=vmin,
        vmax=vmax,
        scalebar_color="white",
        colorbar=True,
        cmap="viridis",
    )

## 8. Optional line profile

The profile is derived from a selected ratio map. Confirm the axis,
index and physical step before using it in a figure or comparison.

In [ ]:
if LINE_PROFILE_RATIO is not None:
    ratio_data = np.asarray(ratio_maps[LINE_PROFILE_RATIO].data)
    start = LINE_PROFILE_START
    stop = min(LINE_PROFILE_STOP, ratio_data.shape[LINE_PROFILE_AXIS])

    if LINE_PROFILE_AXIS == 0:
        fixed_index = min(LINE_PROFILE_INDEX, ratio_data.shape[1] - 1)
        profile = ratio_data[start:stop, fixed_index]
    elif LINE_PROFILE_AXIS == 1:
        fixed_index = min(LINE_PROFILE_INDEX, ratio_data.shape[0] - 1)
        profile = ratio_data[fixed_index, start:stop]
    else:
        raise ValueError("LINE_PROFILE_AXIS must be 0 or 1.")

    if LINE_PROFILE_STEP_NM is None:
        distance = np.arange(profile.size)
        x_label = "Pixel"
    else:
        distance = np.arange(profile.size) * LINE_PROFILE_STEP_NM
        x_label = "Distance (nm)"

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(distance, profile, marker="o", markersize=3)
    ax.set_xlabel(x_label)
    ax.set_ylabel(LINE_PROFILE_RATIO.replace("_", " "))
    ax.grid(alpha=0.3)
    fig.tight_layout()

## 9. Optional export

Export is disabled by default. When enabled, outputs are written to a
repository-relative folder; no directory is cleared or recursively
deleted.

In [ ]:
if SAVE_RESULTS:
    composition_dir = OUTPUT_DIR / "composition_maps"
    ratio_dir = OUTPUT_DIR / "ratio_maps"
    composition_dir.mkdir(parents=True, exist_ok=True)
    ratio_dir.mkdir(parents=True, exist_ok=True)

    for line, map_signal in composition_by_line.items():
        map_signal.save(
            composition_dir / f"{line}_atomic_percent.hspy",
            overwrite=True,
        )

    for name, ratio_signal in ratio_maps.items():
        ratio_signal.save(
            ratio_dir / f"{name}.hspy",
            overwrite=True,
        )

    print("Results written to:", OUTPUT_DIR)

## Interpretation and limitations

PCA reconstruction suppresses noise but can also remove weak or spatially
localized spectral features. Choose components using the scree plot,
factors, loadings and reconstructed spectra—not only visual smoothness.

Cliff–Lorimer results depend on detector calibration, absorption,
specimen thickness, peak overlap, background selection and the supplied
k-factors. Ratio maps additionally amplify uncertainty when denominator
signals are weak; the thresholds above are safeguards, not substitutes
for uncertainty analysis or independent chemical validation.